In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('splits/imdb_ratings_fold4.csv')
df

In [ ]:
df = pd.read_parquet('data/imdb.parquet')

In [ ]:
df

In [ ]:
df = df.sample(n=min(50, df.shape[0]), random_state=42).reset_index(drop=True)

In [ ]:
from transformers import pipeline

# Load the end-to-end ABSA pipeline
absa_pipeline = pipeline(
    "token-classification",
    model="yangheng/deberta-v3-base-end2end-absa",
    aggregation_strategy="simple" # Groups sub-tokens into whole words
)
aspects = []
for text in df['review'].to_list():
    # Run inference
    predictions = absa_pipeline(text)

    # Post-process the output to make it readable
    item_aspect = []
    for pred in predictions:
        label = pred["entity_group"]
        if label == "O":
            continue

        # Labels look like 'B-ASP-Positive' or 'I-ASP-Negative'
        sentiment = label.split("-")[-1]

        item_aspect.append({
            "aspect": pred["word"],
            "sentiment": sentiment,
            "score": round(pred["score"], 4)
        })

    aspects.append(item_aspect)

In [ ]:
df_own = pd.read_csv('data/new_aspects_sent.csv')

In [ ]:
item = 2
print(df.loc[item]['review'])
aspects[item]

In [ ]:
df_own[df_own['review_idx'] == item]

In [ ]:
df

In [ ]:
df_own